In [ ]:
import pandas as pd
import os 
import numpy as np 
from PIL import Image

import torch
import torch.nn.functional as F
import cv2

In [ ]:
def get_cosine_similarity(t1, t2):
    if t1.shape[0] != 1 or len(t1.shape) != 2:
        print(t1.shape)
    
    if t2.shape[0] != 1 or len(t2.shape) != 2:
        print(t2.shape)

    assert len(t1.shape) == 2
    assert len(t2.shape) == 2
    assert t1.shape[0] == 1
    assert t2.shape[0] == 1

    return F.cosine_similarity(t1, t2).item()

def return_image_set(df):
    return set(
        set(df['instance_i']).union(set(df['instance_j']))
    )

In [ ]:
# NOTE can read images from a folder or here we use the dataframe itself to load the image set (and add new features/columns sequentially)
cats = ["airplane", "bed", "bottle", "camera", "car", "chair"]
rootdir = "../data"

df = pd.read_csv(os.path.join(rootdir, "shapenet_sim_data.csv"))

In [ ]:
print(df.shape)
print()
print(df.columns)

In [ ]:
print(df['dataset'].unique())

In [ ]:
all_imgs = return_image_set(df)
print(len(all_imgs))

In [ ]:
clean_set = set()

for img in all_imgs:
    if len(img.split('_')) > 3:
        clean_set.add(img + '_double')
        raise
    else: 
        clean_set.add(img)

assert len(all_imgs) == len(clean_set)

In [ ]:
print(all_imgs)

In [ ]:
@torch.inference_mode()
def extract_PE(target_imgs):
    import core.vision_encoder.pe as pe
    import core.vision_encoder.transforms as transforms
    """
    https://github.com/facebookresearch/perception_models/tree/main
    """
    
    device = "cuda:1" if torch.cuda.is_available() else "cpu"
    model = pe.VisionTransformer.from_config("PE-Spatial-L14-448", pretrained=True).to(device)

    preprocess = transforms.get_image_transform(model.image_size)
    
    base_folder = "/your/path/datasets"

    all_features = {}
    
    @torch.inference_mode()
    def fwd(target_img):
        assert target_img.size[0] >= 448, "image resolution is not high enough for PE-Spatial"
        assert target_img.size[1] >= 448, "image resolution is not high enough for PE-Spatial"

        with torch.autocast(device_type="cuda", enabled=False):
            inputs = preprocess(target_img).unsqueeze(0).to(device)
            image_features, *_ = model(inputs)
        return image_features.cpu()
    
    for img_name in target_imgs:
        splitted = img_name.split('_')
        if "top" in splitted[1]:
            suffix = "top"
        elif "other" in splitted[1]:
            suffix = "other"
        else:
            raise

        cat = splitted[0]
        assert cat in cats

        if len(splitted) <= 3:
            img_path = os.path.join(base_folder, "shapeNet_custom_random", cat, "frames", suffix, img_name)
            image = Image.open(img_path).convert("RGB")
            all_features[img_name] = fwd(target_img=image)
        else:
            raise

    print(f"Extracted features for {len(all_features)} images.")

    return all_features

In [ ]:
@torch.inference_mode()
def extract_CLIP(target_imgs):
    import clip

    device = "cuda:1" if torch.cuda.is_available() else "cpu"
    model, preprocess = clip.load("ViT-B/32", device=device)
    
    base_folder = "/your/path/datasets"

    all_features = {}
    
    @torch.inference_mode()
    def fwd(target_img):

        with torch.autocast(device_type="cuda", enabled=False):
            inputs = preprocess(target_img).unsqueeze(0).to(device)
            outputs = model.encode_image(inputs)
        return outputs.cpu()
    
    for img_name in target_imgs:
        splitted = img_name.split('_')
        if "top" in splitted[1]:
            suffix = "top"
        elif "other" in splitted[1]:
            suffix = "other"
        else:
            raise

        cat = splitted[0]
        assert cat in cats

        if len(splitted) <= 3:
            img_path = os.path.join(base_folder, "shapeNet_custom_random", cat, "frames", suffix, img_name)
            image = Image.open(img_path).convert("RGB")
            all_features[img_name] = fwd(target_img=image)
        else:       
            raise

    print(f"Extracted features for {len(all_features)} images.")

    return all_features

In [ ]:
def extract_RGB_hist(target_imgs):
    
    base_folder = "/your/path/datasets"

    all_features = {}
    
    for img_name in target_imgs:
        splitted = img_name.split('_')
        if "top" in splitted[1]:
            suffix = "top"
        elif "other" in splitted[1]:
            suffix = "other"
        else:
            raise

        cat = splitted[0]
        assert cat in cats

        if len(splitted) <= 3:
            img_path = os.path.join(base_folder, "shapeNet_custom_random", cat, "frames", suffix, img_name)
            image = cv2.imread(img_path)
            image = cv2.resize(image, (224, 224))
            # extract a 3D RGB color histogram from the image,
            # using 8 bins per channel, normalize, and update
            # the index
            hist = cv2.calcHist([image], [0, 1, 2], None, [8, 8, 8],
                [0, 256, 0, 256, 0, 256])
            hist = cv2.normalize(hist, hist).flatten()

            all_features[img_name] = hist
        else:
            raise

    print(f"Extracted features for {len(all_features)} images.")

    return all_features

In [ ]:
# PE_feats = extract_PE(clean_set)
# PE_feats = extract_RGB_hist(clean_set)
# PE_feats = extract_GIST(clean_set)
PE_feats = extract_CLIP(clean_set)

In [ ]:
print(len(PE_feats))
print(PE_feats['airplane_top0_170.jpg'].shape)

In [ ]:
def router(r, feats, i):
    idx = "instance_i" if i == True else "instance_j"
    filename = r[idx]

    def need_unsqueeze(_tensor):
        if (_tensor.shape == (1, 1024)):
            return _tensor 
        elif (_tensor.shape == (1024,)):
            return _tensor.unsqueeze(0)
        else:
            raise
    
    if len(filename.split('_')) <= 3:
        t = feats[filename][0, :]
        ret = need_unsqueeze(t)
    else:
        raise

    return ret

def RGB_router(r, feats, i):
    idx = "instance_i" if i == True else "instance_j"
    filename = r[idx]
    
    if len(filename.split('_')) <= 3:
        t = feats[filename]
    else:
        raise

    return t

In [ ]:
rootdir = "../data"

df["clip_cos_sim"] = df.apply(
    # lambda row: get_cosine_similarity(router(row, PE_feats, i=True), router(row, PE_feats, i=False)), # NOTE PE
    lambda row: get_cosine_similarity(PE_feats[row['instance_i']], PE_feats[row['instance_j']]), # NOTE CLIP
    # lambda row: cv2.compareHist( # NOTE RGB hist
    #         RGB_router(row, PE_feats, i=True), 
    #         RGB_router(row, PE_feats, i=False), 
    #         cv2.HISTCMP_CORREL
    #     ), 
    axis=1
)
df.to_csv(os.path.join(rootdir, "shapenet_sim_data.csv"), index=False)